<a href="https://colab.research.google.com/github/dcow12/Data-Cleaning-for-DS-Appropriations-R3-Funding/blob/main/DS_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning
**Summary**

This notebook inputs raw CSV files containing funding requests, standardizes the file, reformats the strings, removes duplicates, and an audit to showcase where incomplete data is.

**what it does:**
* Ignores  headers and enforces certain constraints if typed wrong
* Cleans `$`, `%`, and standardizes `Y/N` inputs
* Identifies and reports rows where a financial deduction occurred but no justification was provided.
* Identifies missing critical amounts.
* Outputs a standardized CSV.

In [15]:
import pandas as pd
import numpy as np
import io
import re
from google.colab import files
from IPython.display import display, HTML


print("If seen, library has been imported/downloaded.")

If seen, library has been imported/downloaded.


In [16]:
uploaded = files.upload()


filename = list(uploaded.keys())[0]
print(f"File '{filename}' uploaded.")

Saving S26 - RR3(CIO Funding Request List).csv to S26 - RR3(CIO Funding Request List) (2).csv
File 'S26 - RR3(CIO Funding Request List) (2).csv' uploaded.


##This will be based on the Round 3 Funding:
The top columns from the recent CSV file from the Round 3 will be shown to update. Previous was round 2, but recently updated although tags are pretty much the same.
This also filters out any other columns that aren't important and will exit if header not found. Can be fixed by just modifying Target Columns and Values.

In [17]:
TARGET_COLUMNS = [
    'CIO Name', 'CIO Category', 'Email', 'Contact Name', 'Supplier Number',
    'Activity FDM Worktag', 'Primary Reviewer', 'Secondary Reviewer',
    'Primary Done? (type Y/N)', 'Secondary Done? (type Y/N)',
    'Needs Email? (type Y/N)', 'Needs Hearing? (type Y/N)',
    'Allocation Entered + Request Returned (type Y/N)',
    'Scotty + Minseo Approved (type Y/N)', 'Comments',
    'Amount Requested', 'Amount Approved', 'Amount Allocated',
    'Approval Rate', 'Allocation Rate', 'Amount Entered into Presence'
]


raw_bytes = uploaded[filename]
df_raw = pd.read_csv(io.BytesIO(raw_bytes), header=None)


header_row_index = 0
for idx, row in df_raw.iterrows():
    if 'CIO Name' in str(row.values) or 'CIO Category' in str(row.values):
        header_row_index = idx
        break

print(f"Headers on Row {header_row_index + 1} of your CSV.")


df_raw = pd.read_csv(io.BytesIO(raw_bytes), header=header_row_index)

df_raw.columns = df_raw.columns.str.strip()


for col in TARGET_COLUMNS:
    if col not in df_raw.columns:
        print(f"'{col}' not found in CSV. Blank column has been placed in its steed.")
        df_raw[col] = np.nan

df = df_raw[TARGET_COLUMNS].copy()

df = df.dropna(subset=['CIO Name', 'Amount Requested'], how='all')

print(f"Data loaded. Found {len(df)} rows of data.")

Headers on Row 3 of your CSV.
Data loaded. Found 108 rows of data.


This is the cleaning part, changes the values into readable types and then notifies any errors.
##WILL STRIP SOME OF THE VALUES SUCH AS THE PERCENTAGE CURRENCY.
* For example will turn $4,000.00 into 4,000.0

In [18]:
initial_rows = len(df)
df = df.drop_duplicates()
dropped_dupes = initial_rows - len(df)


def clean_currency(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float)): return x
    cleaned = re.sub(r'[^\d.]', '', str(x))
    return float(cleaned) if cleaned else np.nan

def clean_percentage(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float)): return x
    cleaned = re.sub(r'[^\d.]', '', str(x))
    return float(cleaned) / 100 if cleaned else np.nan

def clean_boolean_text(x):
    if pd.isna(x): return np.nan
    text = str(x).strip().upper()
    return 'Y' if text in ['Y', 'YES', 'TRUE'] else ('N' if text in ['N', 'NO', 'FALSE'] else text)



currency_cols = ['Amount Requested', 'Amount Approved', 'Amount Allocated', 'Amount Entered into Presence']
for col in currency_cols:
    df[col] = df[col].apply(clean_currency)

percentage_cols = ['Approval Rate', 'Allocation Rate']
for col in percentage_cols:
    df[col] = df[col].apply(clean_percentage)

yn_cols = [col for col in df.columns if '(type Y/N)' in col]
for col in yn_cols:
    df[col] = df[col].apply(clean_boolean_text)


df['CIO Name'] = df['CIO Name'].apply(lambda x: str(x).strip() if pd.notna(x) else np.nan)
df['Email'] = df['Email'].apply(lambda x: str(x).strip().lower() if pd.notna(x) else np.nan)

print(f"{dropped_dupes} duplicate rows removed.")
print("Data types reformatted (Currencies and Percentages converted to floats).")



0 duplicate rows removed.
Data types reformatted (Currencies and Percentages converted to floats).


In [19]:
print("Fails/Errors in file\n")
audit_issues_found = False

for index, row in df.iterrows():
    cio_name = row['CIO Name'] if pd.notna(row['CIO Name']) else f"Row {index + 2} (Unknown CIO)"
    issues = []

    # 1. Check for missing amounts
    if pd.isna(row['Amount Requested']) or pd.isna(row['Amount Allocated']):
         issues.append("Missing 'Amount Requested' or 'Amount Allocated'.")

    # 2. Check logic: If money was cut, is there a reason provided?
    else:
        req_amt = float(row['Amount Requested'])
        alloc_amt = float(row['Amount Allocated'])

        # If allocated is less than requested, a comment/reason MUST exist
        if alloc_amt < req_amt:
            if pd.isna(row['Comments']) or str(row['Comments']).strip() == "":
                difference = req_amt - alloc_amt
                issues.append(f"Funding reduced by ${difference:,.2f}, no reason/comment provided.")

    # 3. Check for missing Email
    if pd.isna(row['Email']):
        issues.append("Missing contact info.")

    # Print issues to console if any exist for this row
    if issues:
        audit_issues_found = True
        print(f"CIO: {cio_name}:")
        for issue in issues:
            print(f"   - {issue}")
        print("-" * 50)

if not audit_issues_found:
    print("No missing data or logical errors found.")

Fails/Errors in file

CIO: Shoes 4 Hope:
   - Funding reduced by $630.10, no reason/comment provided.
--------------------------------------------------
CIO: Journal of Law and Politics:
   - Funding reduced by $417.27, no reason/comment provided.
--------------------------------------------------
CIO: Anthropomorphics Within University:
   - Funding reduced by $471.64, no reason/comment provided.
--------------------------------------------------
CIO: Orthodox Christian Campus Ministries:
   - Funding reduced by $361.93, no reason/comment provided.
--------------------------------------------------
CIO: Undergraduate Women in Law at UVA:
   - Funding reduced by $51.72, no reason/comment provided.
--------------------------------------------------
CIO: Frugal Innovations in Sustainable Healthcare:
   - Funding reduced by $468.98, no reason/comment provided.
--------------------------------------------------
CIO: 180 Degrees Consulting at UVA:
   - Funding reduced by $4,239.10, no reaso

Everything from here on out is basically just downloaded the file and showcasing. The main purpose is to show flaws in the areas that can be fixed, but if you want the cleaned file, run the part below to get it, although probably can't be used as it has already converted most the parts into floats.

In [20]:
print("Cleaned Data head: ")
display(df.head())


output_filename = "Cleaned_CIO_Funding_Data.csv"
df.to_csv(output_filename, index=False)

print(f"\nFile '{output_filename}' ouptut.")
files.download(output_filename)

Cleaned Data head: 


,CIO Name,CIO Category,Email,Contact Name,Supplier Number,Activity FDM Worktag,Primary Reviewer,Secondary Reviewer,Primary Done? (type Y/N),Secondary Done? (type Y/N),...,Needs Hearing? (type Y/N),Allocation Entered + Request Returned (type Y/N),Scotty + Minseo Approved (type Y/N),Comments,Amount Requested,Amount Approved,Amount Allocated,Approval Rate,Allocation Rate,Amount Entered into Presence
0,The Kendo Club,Club Sport,kwq5ex@virginia.edu,Nuri Han,SPL-41243,NaN,Lila Gerow,Avery,Y,Y,...,NaN,Y,NaN,Denied: CIO did not provide quantity for line ...,4875.50,4694.77,2068.17,0.9629,0.4405,2068.17
1,Mock Trial at the University of Virginia,Academic & Professional,zbm7ax@virginia.edu,Trevor McMillan,SPL-25855,NaN,Lila Gerow,Avery,Y,Y,...,NaN,Y,NaN,Approved,5824.94,0.00,0.00,0.0000,0.0000,0.00
2,Be Somebody,Social & Hobby,ttd8us@virginia.edu,Annie Zhao,Supplier number still pending,NaN,Lila Gerow,Avery,Y,Y,...,NaN,Y,NaN,Approved,1417.48,1191.21,705.17,0.8404,0.5920,705.17
3,Tau Beta Sigma,"Public Service,Visual & Performing Arts",qwv7qc@virginia.edu,Meron Zebro,SPL-46668,NaN,Lila Gerow,Avery,Y,Y,...,NaN,Y,NaN,Approved,852.00,727.00,449.85,0.8533,0.6188,449.85
4,Biokind Analytics at UVA,Academic & Professional,sbt9sh@virginia.edu,Eliza Baddar,SPL-00780,NaN,Lila Gerow,Avery,Y,Y,...,NaN,Y,NaN,Approved,194.90,150.00,97.50,0.7696,0.6500,97.50



File 'Cleaned_CIO_Funding_Data.csv' ouptut.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>